# 🎵 SoundPulse AI — Обучение модели v2 (`MusicAudioNetV2`)

**Что нового в v2:**
- **5 реальных вайбов (настроений):**
  1. `Драйв / Энергия` (рок, металл, агрессивный рэп/фонк, спорт)
  2. `Вечеринка / Танцы` (клубняк, EDM, хаус, бодрый поп, диско)
  3. `Чилл / Отдых` (расслабляющая, акустика, регги, лаунж, мягкий поп)
  4. `Лирика / Грусть` (грустные, романтичные, эмоциональные баллады и вокал)
  5. `Фокус / Концентрация` (инструментал, классика, эмбиент, спокойный джаз, lo-fi)
- **12 современных жанров:** `blues`, `classical`, `country`, `electronic`, `folk`, `hiphop`, `jazz`, `lofi_ambient`, `metal`, `pop`, `reggae`, `rock`.
- **Учитель — `LAION-CLAP` (`larger_clap_music_and_speech`):** предобученная на сотнях тысяч треков аудио-текстовая нейросеть размечает датасет (~8,000–9,000 треков FMA-Small + GTZAN) мягкими вероятностями по 5 реальным вайбам через ансамбль описаний.
- **Студент — `EfficientNet-B0 + Mean/Std Pooling`:** компактная модель (~5 МБ в INT8, ~18 МБ FP32), инференс ~15 мс на CPU, **100% совместимый вход** (`mel_spectrogram [1, 1, 128, 313]`) с C#-плеером `SoundPulse.exe`.

### Как запустить в Kaggle
1. **Notebook Options → Accelerator:** выбери **GPU T4 x2** (или **GPU T4 / P100**).
2. **Notebook Options → Internet:** включи **On**.
3. *(Опционально)* Можно подключить датасет GTZAN через **+ Add Input** (`gtzan-dataset-music-genre-classification`) — тогда он не будет качаться из сети. **FMA-Small** ноутбук скачает сам автоматически (~45 сек).
4. Нажми **Run All** (~12–18 минут на T4).
5. В конце скачай **`music_model_v2_export.zip`** из `/kaggle/working/` и распакуй в `D:\Projects\my_tests\music_player\ml\export\`.


In [ ]:
!pip install -q timm transformers onnx onnxruntime librosa soundfile

import os
import gc
import json
import math
import time
import zipfile
import tarfile
import random
import shutil
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import timm
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import ClapModel, ClapProcessor
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 1. Таксономия v2 (5 вайбов + 12 жанров) и параметры Mel-спектрограммы
Параметры Mel-спектрограммы **1-в-1 совпадают** с `app_csharp/AI/MelSpectrogram.cs`.


In [ ]:
SAMPLE_RATE = 16000
CLAP_SR = 48000
CHUNK_DURATION_SEC = 10.0
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION_SEC)  # 160,000
CLAP_CHUNK_SAMPLES = int(CLAP_SR * CHUNK_DURATION_SEC) # 480,000
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512
TARGET_FRAMES = CHUNK_SAMPLES // HOP_LENGTH + 1        # 313

GENRES = [
    "blues",
    "classical",
    "country",
    "electronic",
    "folk",
    "hiphop",
    "jazz",
    "lofi_ambient",
    "metal",
    "pop",
    "reggae",
    "rock",
]

VIBES = [
    "Драйв / Энергия",       # 0
    "Вечеринка / Танцы",     # 1
    "Чилл / Отдых",          # 2
    "Лирика / Грусть",       # 3
    "Фокус / Концентрация",  # 4
]

# Prompt ensembles for LAION-CLAP zero-shot teacher (4 diverse descriptions per vibe)
VIBE_PROMPTS = {
    0: [
        "aggressive intense high energy rock metal workout driving powerful music",
        "fast distorted electric guitar heavy drums energetic adrenaline pumping song",
        "hard hitting aggressive hip hop trap phonk intense bass powerful beat",
        "fierce high energy fast tempo motivating gym workout music",
    ],
    1: [
        "upbeat dance party club electronic house disco catchy groovy music",
        "fun energetic pop dance song with bouncy four on the floor beat for dancing",
        "festive vibrant club dancefloor electronic pop disco rhythm",
        "cheerful upbeat reggaeton latin pop dance party track",
    ],
    2: [
        "chill relaxing laid back mellow warm acoustic lounge reggae music",
        "easy listening breezy sunny relaxed soft pop groove",
        "calm cozy chillout soft rhythm relaxing weekend vibe",
        "warm smooth acoustic guitar relaxed pleasant chill song",
    ],
    3: [
        "sad emotional melancholic slow romantic lyrical ballad heartbreak song",
        "sorrowful longing soulful vocal ballad with emotional singing and minor chords",
        "heartfelt nostalgic slow pop ballad dramatic emotional vocals about love and pain",
        "melancholic lonely bittersweet tearful slow lyrical vocal music",
    ],
    4: [
        "calm instrumental classical piano ambient jazz study focus background music without vocals",
        "peaceful solo piano acoustic instrumental music for deep concentration and reading",
        "minimal atmospheric ambient cinematic instrumental soundscape without singing",
        "quiet smooth instrumental jazz study lo-fi beats for focus",
    ],
}

GENRE_PROMPTS = {
    "blues": [
        "blues music with expressive electric guitar licks and soulful rhythm",
        "classic twelve bar blues guitar and harmonica song",
    ],
    "classical": [
        "classical orchestral symphony violin cello piano chamber music",
        "romantic baroque classical instrumental piano piece",
    ],
    "country": [
        "country music with acoustic guitar banjo pedal steel and folk vocals",
        "american country western acoustic song",
    ],
    "electronic": [
        "electronic EDM house techno synthwave club synthesizer dance music",
        "modern electronic dance music with synthesizers and drum machine",
    ],
    "folk": [
        "acoustic folk indie singer songwriter guitar storytelling music",
        "traditional acoustic folk melody with acoustic strings",
    ],
    "hiphop": [
        "hip hop rap trap music with rhythmic vocals 808 bass and drum beat",
        "rap song with hip hop beat and spoken rhythmic flow",
    ],
    "jazz": [
        "jazz music with saxophone trumpet upright bass swing drums and piano",
        "smooth jazz bebop instrumental improvisation",
    ],
    "lofi_ambient": [
        "lo-fi chillhop ambient atmospheric dreamy downtempo instrumental beats",
        "ethereal ambient electronic soundscape calm lo-fi study music",
    ],
    "metal": [
        "heavy metal hard rock with heavily distorted guitars double kick drums and screaming",
        "thrash metal aggressive distorted guitar riffs and intense percussion",
    ],
    "pop": [
        "modern pop song with catchy vocal melody chorus and polished production",
        "mainstream pop vocal ballad or upbeat radio pop music",
    ],
    "reggae": [
        "reggae dub ska music with offbeat guitar skank and deep warm bassline",
        "jamaican reggae rhythm laid back groove",
    ],
    "rock": [
        "rock alternative indie rock song with electric guitars live drums and vocals",
        "classic rock band with guitars bass and energetic drums",
    ],
}

FMA_TO_GENRE_IDX = {
    "Electronic": GENRES.index("electronic"),
    "Folk": GENRES.index("folk"),
    "Hip-Hop": GENRES.index("hiphop"),
    "Pop": GENRES.index("pop"),
    "Rock": GENRES.index("rock"),
}

GTZAN_TO_GENRE_IDX = {
    "blues": GENRES.index("blues"),
    "classical": GENRES.index("classical"),
    "country": GENRES.index("country"),
    "disco": GENRES.index("electronic"),
    "hiphop": GENRES.index("hiphop"),
    "jazz": GENRES.index("jazz"),
    "metal": GENRES.index("metal"),
    "pop": GENRES.index("pop"),
    "reggae": GENRES.index("reggae"),
    "rock": GENRES.index("rock"),
}

print(f"Genres ({len(GENRES)}): {GENRES}")
print(f"Vibes  ({len(VIBES)}): {VIBES}")


## 2. Загрузка датасетов (FMA-Small 8,000 треков + GTZAN 1,000 треков)
Сначала ищем подключённые датасеты в `/kaggle/input/`, а недостающее автоматически скачиваем.


In [ ]:
WORK_DIR = Path("/kaggle/working/soundpulse_v2")
DATA_DIR = WORK_DIR / "data"
CACHE_DIR = WORK_DIR / "cache"
EXPORT_DIR = WORK_DIR / "export"
for d in (DATA_DIR, CACHE_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)


def run_download(url: str, dest: Path, timeout_sec: int = 600) -> bool:
    if dest.exists() and dest.stat().st_size > 1_000_000:
        print(f"  [Skip] Already exists: {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)")
        return True
    print(f"  [Download] {url} -> {dest.name} ...")
    cmd = ["curl", "-L", "--fail", "--retry", "3", "--connect-timeout", "20", "-o", str(dest), url]
    res = subprocess.run(cmd, timeout=timeout_sec)
    return res.returncode == 0 and dest.exists() and dest.stat().st_size > 1_000_000


def find_existing_dir(candidates):
    for root in [Path("/kaggle/input"), DATA_DIR]:
        if not root.exists():
            continue
        for pat in candidates:
            matches = list(root.rglob(pat))
            for m in matches:
                if m.is_dir():
                    return m
    return None


# --- 1. FMA-Small (8,000 modern tracks across 8 genres) ---
fma_audio_dir = find_existing_dir(["fma_small"])
fma_meta_csv = None
for root in [Path("/kaggle/input"), DATA_DIR]:
    if root.exists():
        found = list(root.rglob("tracks.csv"))
        if found:
            fma_meta_csv = found[0]
            break

if fma_audio_dir is None:
    fma_zip = DATA_DIR / "fma_small.zip"
    if run_download("https://os.unil.cloud.switch.ch/fma/fma_small.zip", fma_zip):
        print("  [Extract] fma_small.zip ...")
        with zipfile.ZipFile(fma_zip, "r") as zf:
            zf.extractall(DATA_DIR)
        fma_zip.unlink(missing_ok=True)
        fma_audio_dir = DATA_DIR / "fma_small"

if fma_meta_csv is None:
    meta_zip = DATA_DIR / "fma_metadata.zip"
    if run_download("https://os.unil.cloud.switch.ch/fma/fma_metadata.zip", meta_zip):
        print("  [Extract] fma_metadata.zip (tracks.csv only) ...")
        with zipfile.ZipFile(meta_zip, "r") as zf:
            for member in zf.namelist():
                if member.endswith("tracks.csv"):
                    zf.extract(member, DATA_DIR)
        meta_zip.unlink(missing_ok=True)
        found = list(DATA_DIR.rglob("tracks.csv"))
        if found:
            fma_meta_csv = found[0]

# --- 2. GTZAN (1,000 tracks — optional bonus for Classical/Jazz/Metal/Blues/Reggae/Country) ---
gtzan_dir = find_existing_dir(["genres_original", "genres"])
if gtzan_dir is None or len(list(gtzan_dir.glob("*/*.wav"))) < 500:
    try:
        gtzan_tar = DATA_DIR / "genres.tar.gz"
        if run_download("http://opihi.cs.uvic.ca/sound/genres.tar.gz", gtzan_tar, timeout_sec=120):
            with tarfile.open(gtzan_tar, "r:gz") as tar:
                tar.extractall(DATA_DIR)
            gtzan_tar.unlink(missing_ok=True)
            gtzan_dir = DATA_DIR / "genres"
    except Exception as ex:
        print(f"  [Note] Optional GTZAN download skipped ({ex}). FMA-Small alone is 8,000 tracks!")
        gtzan_dir = None

# --- Collect unified track manifest: (audio_path, gt_genre_idx_or_None, source_name) ---
tracks_manifest = []

if fma_audio_dir is not None and fma_audio_dir.exists():
    fma_genre_map = {}
    if fma_meta_csv is not None and fma_meta_csv.exists():
        try:
            df = pd.read_csv(fma_meta_csv, index_col=0, header=[0, 1])
            subset_col = ("set", "subset")
            genre_col = ("track", "genre_top")
            small_df = df[df[subset_col] == "small"]
            for tid, gname in small_df[genre_col].items():
                if isinstance(gname, str):
                    fma_genre_map[int(tid)] = gname
        except Exception as ex:
            print(f"  [Warn] Could not parse FMA tracks.csv: {ex}")

    mp3_files = sorted(fma_audio_dir.rglob("*.mp3"))
    for p in mp3_files:
        try:
            tid = int(p.stem)
        except ValueError:
            tid = -1
        gname = fma_genre_map.get(tid, "")
        gt_idx = FMA_TO_GENRE_IDX.get(gname, None)
        tracks_manifest.append((p, gt_idx, "fma"))
    print(f"[Dataset] FMA-Small tracks found: {len(mp3_files)}")

if gtzan_dir is not None and gtzan_dir.exists():
    gtzan_count = 0
    for g_name, gt_idx in GTZAN_TO_GENRE_IDX.items():
        folder = gtzan_dir / g_name
        if not folder.exists():
            continue
        for p in sorted(folder.glob("*.wav")):
            tracks_manifest.append((p, gt_idx, "gtzan"))
            gtzan_count += 1
    print(f"[Dataset] GTZAN tracks found: {gtzan_count}")

random.shuffle(tracks_manifest)
print(f"[Dataset] TOTAL tracks ready for CLAP teacher labeling: {len(tracks_manifest)}")
assert len(tracks_manifest) >= 500, "No dataset found! Check Internet=On in Kaggle settings."


## 3. Разметка учителем `LAION-CLAP` + кэширование Mel-спектрограмм (один проход!)
Для каждого трека извлекаем **2 репрезентативных 10-секундных среза** (28% и 66% трека), считаем:
1. Точную Mel-спектрограмму `[128, 313]` в `float16` (сохраняем в memory-mapped файл на диске, чтобы обучение 30 эпох заняло всего ~3 минуты!).
2. Мягкие вероятности **5 вайбов** и **12 жанров** от предобученной модели `laion/larger_clap_music_and_speech`.


In [ ]:
CLAP_MODEL_ID = "laion/larger_clap_music_and_speech"
print(f"[CLAP] Loading teacher model: {CLAP_MODEL_ID} ...")
clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL_ID)
clap_model = ClapModel.from_pretrained(CLAP_MODEL_ID).to(DEVICE).eval()


@torch.no_grad()
def encode_prompt_ensemble(prompt_dict, keys):
    emb_list = []
    for k in keys:
        prompts = prompt_dict[k]
        inputs = clap_processor(text=prompts, return_tensors="pt", padding=True).to(DEVICE)
        text_feat = clap_model.get_text_features(**inputs)
        if not isinstance(text_feat, torch.Tensor): text_feat = text_feat.pooler_output
        text_feat = F.normalize(text_feat, dim=-1)
        mean_feat = F.normalize(text_feat.mean(dim=0, keepdim=True), dim=-1)
        emb_list.append(mean_feat)
    return torch.cat(emb_list, dim=0)


vibe_text_emb = encode_prompt_ensemble(VIBE_PROMPTS, list(range(len(VIBES))))
genre_text_emb = encode_prompt_ensemble(GENRE_PROMPTS, GENRES)
print(f"[CLAP] Text embeddings ready: Vibes={tuple(vibe_text_emb.shape)}, Genres={tuple(genre_text_emb.shape)}")

mel_tf_gpu = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    win_length=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    power=2.0,
).to(DEVICE)
amp_to_db_gpu = T.AmplitudeToDB().to(DEVICE)


def load_two_crops_48k_and_16k(audio_path: Path):
    """Loads audio once and returns two 10s crops at 48 kHz (for CLAP) and 16 kHz (for Student Mel)."""
    try:
        y48, _ = librosa.load(str(audio_path), sr=CLAP_SR, mono=True)
        if y48 is None or len(y48) < CLAP_SR * 3:
            return None
    except Exception:
        return None

    total = len(y48)
    if total >= CLAP_CHUNK_SAMPLES * 1.8:
        starts = [
            max(0, min(int(total * 0.28 - CLAP_CHUNK_SAMPLES * 0.5), total - CLAP_CHUNK_SAMPLES)),
            max(0, min(int(total * 0.66 - CLAP_CHUNK_SAMPLES * 0.5), total - CLAP_CHUNK_SAMPLES)),
        ]
    else:
        starts = [max(0, (total - CLAP_CHUNK_SAMPLES) // 2)]

    crops_48k = []
    crops_16k = []
    for st in starts:
        seg = y48[st:st + CLAP_CHUNK_SAMPLES]
        if len(seg) < CLAP_CHUNK_SAMPLES:
            seg = np.pad(seg, (0, CLAP_CHUNK_SAMPLES - len(seg)))
        seg = seg.astype(np.float32, copy=False)
        crops_48k.append(seg)
        seg16 = librosa.resample(seg, orig_sr=CLAP_SR, target_sr=SAMPLE_RATE, res_type="polyphase")
        if len(seg16) < CHUNK_SAMPLES:
            seg16 = np.pad(seg16, (0, CHUNK_SAMPLES - len(seg16)))
        else:
            seg16 = seg16[:CHUNK_SAMPLES]
        crops_16k.append(seg16.astype(np.float32, copy=False))

    return crops_48k, crops_16k


MAX_SAMPLES = len(tracks_manifest) * 2
MEL_MEMMAP_PATH = CACHE_DIR / "mels_fp16.dat"
META_NPZ_PATH = CACHE_DIR / "targets.npz"

VIBE_TEMP = 0.065
GENRE_TEMP = 0.075
GT_BLEND = 0.65

if not META_NPZ_PATH.exists():
    mels_mm = np.memmap(MEL_MEMMAP_PATH, dtype=np.float16, mode="w+", shape=(MAX_SAMPLES, N_MELS, TARGET_FRAMES))
    all_vibe_probs = np.zeros((MAX_SAMPLES, len(VIBES)), dtype=np.float32)
    all_genre_probs = np.zeros((MAX_SAMPLES, len(GENRES)), dtype=np.float32)
    all_track_ids = np.zeros((MAX_SAMPLES,), dtype=np.int32)
    all_gt_genres = np.full((MAX_SAMPLES,), -1, dtype=np.int32)

    write_idx = 0
    BATCH_TRACKS = 32

    with ThreadPoolExecutor(max_workers=4) as pool:
        for b_start in tqdm(range(0, len(tracks_manifest), BATCH_TRACKS), desc="CLAP Teacher + Mel Cache"):
            batch_entries = tracks_manifest[b_start:b_start + BATCH_TRACKS]
            decoded = list(pool.map(lambda e: load_two_crops_48k_and_16k(e[0]), batch_entries))

            batch_48k = []
            batch_16k = []
            batch_meta = []

            for local_i, item in enumerate(decoded):
                if item is None:
                    continue
                crops_48, crops_16 = item
                track_id = b_start + local_i
                gt_g = batch_entries[local_i][1]
                for c48, c16 in zip(crops_48, crops_16):
                    batch_48k.append(c48)
                    batch_16k.append(c16)
                    batch_meta.append((track_id, gt_g))

            if not batch_48k:
                continue

            # 1) Exact torchaudio Log-Mel Spectrograms on GPU
            with torch.no_grad():
                wav16_t = torch.from_numpy(np.stack(batch_16k)).to(DEVICE)
                mel = amp_to_db_gpu(mel_tf_gpu(wav16_t))
                mean = mel.mean(dim=(1, 2), keepdim=True)
                std = mel.std(dim=(1, 2), keepdim=True)
                mel_norm = ((mel - mean) / (std + 1e-6)).cpu().numpy().astype(np.float16)

            # 2) CLAP Teacher soft targets on GPU
            with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                try:
                    clap_in = clap_processor(audio=batch_48k, sampling_rate=CLAP_SR, return_tensors="pt", padding=True)
                except TypeError:
                    clap_in = clap_processor(audios=batch_48k, sampling_rate=CLAP_SR, return_tensors="pt", padding=True)
                clap_in = {k: v.to(DEVICE) for k, v in clap_in.items()}
                audio_out = clap_model.get_audio_features(**clap_in)
                if not isinstance(audio_out, torch.Tensor): audio_out = audio_out.pooler_output
                audio_feat = F.normalize(audio_out.float(), dim=-1)

                vibe_sim = audio_feat @ vibe_text_emb.T
                genre_sim = audio_feat @ genre_text_emb.T

                vibe_p = F.softmax(vibe_sim / VIBE_TEMP, dim=-1).cpu().numpy()
                genre_p = F.softmax(genre_sim / GENRE_TEMP, dim=-1).cpu().numpy()

            for k in range(len(batch_meta)):
                tid, gt_g = batch_meta[k]
                gp = genre_p[k].copy()
                if gt_g is not None:
                    one_hot = np.zeros(len(GENRES), dtype=np.float32)
                    one_hot[gt_g] = 1.0
                    gp = GT_BLEND * one_hot + (1.0 - GT_BLEND) * gp

                mels_mm[write_idx] = mel_norm[k]
                all_vibe_probs[write_idx] = vibe_p[k]
                all_genre_probs[write_idx] = gp
                all_track_ids[write_idx] = tid
                all_gt_genres[write_idx] = -1 if gt_g is None else gt_g
                write_idx += 1

    mels_mm.flush()
    del mels_mm

    np.savez_compressed(
        META_NPZ_PATH,
        count=write_idx,
        vibe_probs=all_vibe_probs[:write_idx],
        genre_probs=all_genre_probs[:write_idx],
        track_ids=all_track_ids[:write_idx],
        gt_genres=all_gt_genres[:write_idx],
    )
    print(f"[Cache] Saved {write_idx} labeled Mel crops!")
else:
    print("[Cache] Reusing existing cached Mel spectrograms & CLAP teacher targets.")

del clap_model, clap_processor, vibe_text_emb, genre_text_emb
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

meta = np.load(META_NPZ_PATH)
N_TOTAL = int(meta["count"])
vibe_probs_all = meta["vibe_probs"]
genre_probs_all = meta["genre_probs"]
track_ids_all = meta["track_ids"]

print(f"\n=== Teacher Label Distribution ({N_TOTAL} crops) ===")
vibe_top = vibe_probs_all.argmax(axis=1)
for i, vname in enumerate(VIBES):
    cnt = int((vibe_top == i).sum())
    print(f"  Vibe {i} ({vname:22s}): {cnt:5d} ({100.0 * cnt / N_TOTAL:5.1f}%)")

print("")
genre_top = genre_probs_all.argmax(axis=1)
for i, gname in enumerate(GENRES):
    cnt = int((genre_top == i).sum())
    print(f"  Genre {i:2d} ({gname:14s}): {cnt:5d} ({100.0 * cnt / N_TOTAL:5.1f}%)")


## 4. Архитектура студента `MusicAudioNetV2` (`EfficientNet-B0` + Mean/Std Pooling)
- Принимает ровно `[B, 1, 128, 313]` (совместимо с `SoundPulse.exe`).
- `Mean + Std Pooling` сохраняет одновременно и средний тембр, и резкость/динамику ритма.
- Обучается через **Soft Cross-Entropy Distillation + SpecAugment + Mixup**.


In [ ]:
unique_tids = np.unique(track_ids_all)
rng = np.random.default_rng(SEED)
rng.shuffle(unique_tids)
split_n = int(len(unique_tids) * 0.85)
train_tids = set(unique_tids[:split_n].tolist())

train_indices = [i for i, tid in enumerate(track_ids_all) if int(tid) in train_tids]
val_indices = [i for i, tid in enumerate(track_ids_all) if int(tid) not in train_tids]
print(f"Train crops: {len(train_indices)} | Val crops: {len(val_indices)}")


class CachedMelDataset(Dataset):
    def __init__(self, indices, is_train: bool = True):
        self.indices = indices
        self.is_train = is_train
        self.mels = np.memmap(MEL_MEMMAP_PATH, dtype=np.float16, mode="r", shape=(MAX_SAMPLES, N_MELS, TARGET_FRAMES))
        self.freq_mask = T.FrequencyMasking(freq_mask_param=18)
        self.time_mask = T.TimeMasking(time_mask_param=36)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_i = self.indices[idx]
        mel = torch.from_numpy(np.array(self.mels[real_i], dtype=np.float32)).unsqueeze(0)
        if self.is_train:
            if random.random() < 0.5:
                shift = random.randint(-40, 40)
                mel = torch.roll(mel, shifts=shift, dims=-1)
            mel = self.time_mask(self.freq_mask(mel))
            if random.random() < 0.3:
                mel = mel + 0.05 * torch.randn_like(mel)

        g_tgt = torch.from_numpy(genre_probs_all[real_i])
        v_tgt = torch.from_numpy(vibe_probs_all[real_i])
        return mel, g_tgt, v_tgt


class MusicAudioNetV2(nn.Module):
    def __init__(self, num_genres: int = len(GENRES), num_vibes: int = len(VIBES), pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=pretrained,
            in_chans=1,
            num_classes=0,
            global_pool="",
        )
        feat_dim = self.backbone.num_features  # 1280
        pooled_dim = feat_dim * 2              # Mean + Std = 2560

        self.norm = nn.LayerNorm(pooled_dim)
        self.dropout = nn.Dropout(0.30)

        self.genre_head = nn.Sequential(
            nn.Linear(pooled_dim, 256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(256, num_genres),
        )
        self.vibe_head = nn.Sequential(
            nn.Linear(pooled_dim, 128),
            nn.SiLU(inplace=True),
            nn.Dropout(0.15),
            nn.Linear(128, num_vibes),
        )

    def forward(self, mel: torch.Tensor):
        fmap = self.backbone(mel)
        mean = fmap.mean(dim=(2, 3))
        var = ((fmap - mean.unsqueeze(-1).unsqueeze(-1)) ** 2).mean(dim=(2, 3))
        std = torch.sqrt(var + 1e-6)
        feat = self.dropout(self.norm(torch.cat([mean, std], dim=1)))
        return self.genre_head(feat), self.vibe_head(feat)


model = MusicAudioNetV2(pretrained=True).to(DEVICE)
print(f"MusicAudioNetV2 params: {sum(p.numel() for p in model.parameters()):,}")


## 5. Обучение студента (30 эпох, Mixup + CosineAnnealing + AMP FP16)
Поскольку Mel-спектрограммы уже закэшированы на диске, все 30 эпох пройдут за ~3–5 минут!


In [ ]:
EPOCHS = 30
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4

train_loader = DataLoader(
    CachedMelDataset(train_indices, is_train=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    CachedMelDataset(val_indices, is_train=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))


def soft_ce_loss(logits: torch.Tensor, target_probs: torch.Tensor) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    return -(target_probs * log_probs).sum(dim=-1).mean()


best_score = 0.0
best_ckpt = WORK_DIR / "best_music_v2.pth"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss = 0.0
    tr_n = 0

    for mel, g_tgt, v_tgt in train_loader:
        mel = mel.to(DEVICE, non_blocking=True)
        g_tgt = g_tgt.to(DEVICE, non_blocking=True)
        v_tgt = v_tgt.to(DEVICE, non_blocking=True)

        if random.random() < 0.4:
            lam = float(np.random.beta(0.4, 0.4))
            perm = torch.randperm(mel.size(0), device=DEVICE)
            mel = lam * mel + (1.0 - lam) * mel[perm]
            g_tgt = lam * g_tgt + (1.0 - lam) * g_tgt[perm]
            v_tgt = lam * v_tgt + (1.0 - lam) * v_tgt[perm]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            g_logits, v_logits = model(mel)
            loss_g = soft_ce_loss(g_logits, g_tgt)
            loss_v = soft_ce_loss(v_logits, v_tgt)
            loss = loss_g + 1.15 * loss_v

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        tr_loss += loss.item() * mel.size(0)
        tr_n += mel.size(0)

    scheduler.step()

    model.eval()
    val_loss = 0.0
    g_correct = 0
    v_correct = 0
    val_n = 0

    with torch.no_grad():
        for mel, g_tgt, v_tgt in val_loader:
            mel = mel.to(DEVICE, non_blocking=True)
            g_tgt = g_tgt.to(DEVICE, non_blocking=True)
            v_tgt = v_tgt.to(DEVICE, non_blocking=True)
            g_logits, v_logits = model(mel)

            loss = soft_ce_loss(g_logits, g_tgt) + 1.15 * soft_ce_loss(v_logits, v_tgt)
            val_loss += loss.item() * mel.size(0)
            g_correct += (g_logits.argmax(dim=-1) == g_tgt.argmax(dim=-1)).sum().item()
            v_correct += (v_logits.argmax(dim=-1) == v_tgt.argmax(dim=-1)).sum().item()
            val_n += mel.size(0)

    g_acc = 100.0 * g_correct / val_n
    v_acc = 100.0 * v_correct / val_n
    combined = 0.45 * g_acc + 0.55 * v_acc

    marker = ""
    if combined > best_score:
        best_score = combined
        torch.save(model.state_dict(), best_ckpt)
        marker = "  ★ BEST"

    print(
        f"Epoch {epoch:02d}/{EPOCHS} [{time.time() - t0:4.1f}s] | "
        f"Train {tr_loss / tr_n:.4f} | Val {val_loss / val_n:.4f} | "
        f"Genre Top1: {g_acc:5.1f}% | Vibe Top1: {v_acc:5.1f}%{marker}"
    )


## 6. Экспорт в ONNX (FP32 + INT8) и упаковка `music_model_v2_export.zip`
Проверяем совпадение PyTorch vs ONNX Runtime и замеряем скорость инференса на CPU.


In [ ]:
best_model = MusicAudioNetV2(pretrained=False)
best_model.load_state_dict(torch.load(best_ckpt, map_location="cpu"))
best_model.eval().cpu()

fp32_onnx = EXPORT_DIR / "music_classifier.onnx"
int8_onnx = EXPORT_DIR / "music_classifier.int8.onnx"
labels_path = EXPORT_DIR / "labels.json"

dummy_input = torch.randn(1, 1, N_MELS, TARGET_FRAMES, dtype=torch.float32)

torch.onnx.export(
    best_model,
    dummy_input,
    str(fp32_onnx),
    input_names=["mel_spectrogram"],
    output_names=["genre_logits", "vibe_logits"],
    dynamic_axes={
        "mel_spectrogram": {0: "batch_size", 3: "time_steps"},
        "genre_logits": {0: "batch_size"},
        "vibe_logits": {0: "batch_size"},
    },
    opset_version=17,
    dynamo=False,
)

onnx.checker.check_model(onnx.load(str(fp32_onnx)))
print(f"[Export] FP32 ONNX valid: {fp32_onnx.stat().st_size / (1024 * 1024):.2f} MB")

quantize_dynamic(
    model_input=str(fp32_onnx),
    model_output=str(int8_onnx),
    weight_type=QuantType.QInt8,
)
print(f"[Export] INT8 ONNX ready: {int8_onnx.stat().st_size / (1024 * 1024):.2f} MB")

with torch.no_grad():
    pt_g, pt_v = best_model(dummy_input)

sess = ort.InferenceSession(str(fp32_onnx), providers=["CPUExecutionProvider"])
t0 = time.perf_counter()
for _ in range(10):
    ort_g, ort_v = sess.run(None, {"mel_spectrogram": dummy_input.numpy()})
avg_ms = (time.perf_counter() - t0) * 100.0
max_diff = max(float(np.abs(pt_g.numpy() - ort_g).max()), float(np.abs(pt_v.numpy() - ort_v).max()))
print(f"[Verify] PyTorch vs ONNX max diff: {max_diff:.6f} | CPU latency: {avg_ms:.1f} ms")

metadata = {
    "model_name": "MusicAudioNetV2",
    "version": 2,
    "sample_rate": SAMPLE_RATE,
    "chunk_duration_sec": CHUNK_DURATION_SEC,
    "chunk_samples": CHUNK_SAMPLES,
    "n_mels": N_MELS,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "genres": GENRES,
    "vibes": VIBES,
}
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

zip_out = Path("/kaggle/working/music_model_v2_export.zip")
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(fp32_onnx, arcname="music_classifier.onnx")
    zf.write(int8_onnx, arcname="music_classifier.int8.onnx")
    zf.write(labels_path, arcname="labels.json")

print(f"\n✅ ГОТОВО! Скачай архив: {zip_out} ({zip_out.stat().st_size / (1024 * 1024):.2f} MB)")
print("Распакуй его в D:\\Projects\\my_tests\\music_player\\ml\\export\\ (с заменой файлов).")
